<a href="https://colab.research.google.com/github/Quentalheitor/Flyrank_Heitor_Quental_ML_Track/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Grain:** one row = one **content page owned by a specific client, observed on one daily snapshot date**.
Uniquely identified by the composite key `(client_hash_id, content_hash_id, report_date)`.

**Time window:** `month = '2026-03'`, i.e. `report_date` between `2026-03-01` and `2026-03-31` inclusive
(the mid-panel development window for this assignment).

**Source tables:** `fact_content_daily_performance` (partitioned by `month`) joined to `dim_content.parquet`
on `content_hash_id`. The fact table carries the daily metrics; the dim table carries the (mostly
slower-moving) content/client attributes.

**Window caveat:** `fact_content_daily_performance` is a genuine daily panel — one row per day.
Whether `dim_content` is versioned per day the same way, or is a single current-state snapshot,
is unconfirmed. If it's the latter, `word_count` and `backlinks` reflect content state at
extraction time, not necessarily the true state on each historical `report_date` — the standard
risk of joining a panel to a dimension table with a different window. Checked in Section 4.

The grain and window claims above are checked with DuckDB queries in Section 3 (composite-key
duplicate check, row count, `MIN`/`MAX(report_date)`).

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (identifiers / bookkeeping, not fed to a model as a predictive signal):
`client_hash_id`, `content_hash_id`, `report_date`, `month`, `is_published`, `is_deleted`,
`gsc_data_available`.

**Feature** (knowable at decision time T, used to predict the label):
`gsc_clicks`, `gsc_impressions`, `ga4_total_engagement_sec`, `word_count`, `backlinks`.

**Label / proxy** (the target we are trying to predict):
`is_high_ai_spike` — a binary 1/0 flag computed as

```
(sessions_ai / (gsc_clicks + 1.0) > 0.35) AND (sessions_ai >= 5)
```

It flags pages whose AI-referral traffic is disproportionately large relative to their normal
search traffic, i.e. a "spike" rather than steady baseline AI citation volume.

**Excluded** — `sessions_ai` and every individual AI sub-metric (`ai_chatgpt`, `ai_perplexity`,
`ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`).
**Why:** these columns are the literal inputs used to construct the label. Feeding them to a model
as features would let the model "see" the answer inside the question — direct target leakage. A
quick empirical demonstration of this is included as the leakage "trap" in Section 3.

In [2]:
CONTEXT_FIELDS = [
    "client_hash_id", "content_hash_id", "report_date", "month",
    "is_published", "is_deleted", "gsc_data_available",
]

FEATURE_FIELDS = [
    "gsc_clicks", "gsc_impressions", "ga4_total_engagement_sec",
    "word_count", "backlinks",
]

LABEL_FIELD = "is_high_ai_spike"

EXCLUDED_FIELDS = [
    "sessions_ai",
    "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
]

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four checks, in order: (1) the composite key is truly unique — zero duplicate groups, (2) the row
count and date range match the stated `2026-03` window, (3) how many rows survive the
publish/deleted/GSC-availability filters, plus the NULL rate on each of the 5 approved features
(the missing-values check, folded into the same query), and (4) a feature frame built strictly
from the approved feature list, plus a deliberate "trap" that re-adds `sessions_ai` to show what
leakage looks like in a metric before we agree never to do it again.

In [3]:
grain_check = con.execute(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.report_date, COUNT(*) AS n_rows
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate grain groups found: {len(grain_check)}")
assert len(grain_check) == 0
grain_check.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain groups found: 0


,client_hash_id,content_hash_id,report_date,n_rows


In [4]:
counts_check = con.execute(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_report_date, MAX(report_date) AS max_report_date
    FROM read_parquet('{FACT_PATH}')
""").df()

counts_check

,n_rows,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [5]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS n_surviving_rows,
        AVG(CASE WHEN f.gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_clicks,
        AVG(CASE WHEN f.gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_impressions,
        AVG(CASE WHEN f.ga4_total_engagement_sec IS NULL THEN 1.0 ELSE 0 END) AS pct_null_ga4_total_engagement_sec,
        AVG(CASE WHEN c.word_count IS NULL THEN 1.0 ELSE 0 END) AS pct_null_word_count,
        AVG(CASE WHEN c.backlinks IS NULL THEN 1.0 ELSE 0 END) AS pct_null_backlinks
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

n_total = counts_check["n_rows"].iloc[0]
n_surviving = availability_check["n_surviving_rows"].iloc[0]

print(f"Total rows:      {n_total}")
print(f"Surviving rows:  {n_surviving}")
print(f"Dropped by filters: {n_total - n_surviving} ({(n_total - n_surviving) / n_total:.1%} of rows)")
for col in FEATURE_FIELDS:
    pct = availability_check[f"pct_null_{col}"].iloc[0]
    print(f"NULL rate on {col:<25s}: {pct:.2%}")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows:      9841378
Surviving rows:  3609108
Dropped by filters: 6232270 (63.3% of rows)
NULL rate on gsc_clicks               : 0.00%
NULL rate on gsc_impressions          : 0.00%
NULL rate on ga4_total_engagement_sec : 42.35%
NULL rate on word_count               : 33.35%
NULL rate on backlinks                : 39.07%


,n_surviving_rows,pct_null_gsc_clicks,pct_null_gsc_impressions,pct_null_ga4_total_engagement_sec,pct_null_word_count,pct_null_backlinks
0,3609108,0.0,0.0,0.423469,0.333528,0.390694


### Five-feature frame + the "knowable at T" check

`word_count` and `backlinks` live on `dim_content` (page-level structural attributes), not on
the fact table, so each feature below is resolved to its correct source table. Because that join
is at the content level, `word_count` and `backlinks` repeat identically across every day of a
given `content_hash_id` in this panel — they're read here directly per row, never summed across
days, avoiding the classic repeated-context-column double-counting trap.

**Why each feature is knowable at the decision moment T** (the daily snapshot date):

- `gsc_clicks` — a Search Console metric already recorded for the day of the snapshot; no
  future information required.
- `gsc_impressions` — same source and timing as clicks; observed as of T, not after.
- `ga4_total_engagement_sec` — GA4 engagement accumulated up to the snapshot date, not a
  forward-looking aggregate.
- `word_count` — a structural property of the published page as it exists at T.
- `backlinks` — the backlink count as crawled/measured as of the snapshot; a point-in-time
  count, not something that depends on what happens after T.

In [6]:
fact_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_PATH}') LIMIT 0").df()["column_name"].tolist()
dim_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{DIM_PATH}') LIMIT 0").df()["column_name"].tolist()

def qualify(col):
    if col in fact_schema:
        return f"f.{col}"
    if col in dim_schema:
        return f"c.{col}"
    raise ValueError(f"'{col}' not found in fact or dim schema")

feature_select = ", ".join(f"{qualify(col)} AS {col}" for col in FEATURE_FIELDS)

frame = con.execute(f"""
    SELECT
        {feature_select},
        f.sessions_ai,
        CASE
            WHEN (f.sessions_ai / (f.gsc_clicks + 1.0) > 0.35) AND (f.sessions_ai >= 5)
            THEN 1 ELSE 0
        END AS {LABEL_FIELD}
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

frame = frame.dropna(subset=FEATURE_FIELDS + [LABEL_FIELD])
frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_clicks,gsc_impressions,ga4_total_engagement_sec,word_count,backlinks,sessions_ai,is_high_ai_spike
9497,0,1,0,2971,0,0,0
9498,0,1,0,3109,0,0,0
9499,0,1,0,2934,0,0,0
9500,0,1,0,2449,0,0,0
9501,0,2,0,2719,0,0,0


### The trap

Add the one label-derived column (`sessions_ai`) into the feature set on purpose, fit a quick
`LogisticRegression`, and watch the score jump toward a perfect score. Then drop it and keep the
honest number.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

X_honest = frame[FEATURE_FIELDS]
X_leaky = frame[FEATURE_FIELDS + ["sessions_ai"]]
y = frame[LABEL_FIELD]

def fit_and_score(X, y, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    preds = clf.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, preds)
    print(f"{label:>7s} -> AUC: {auc:.4f}  |  Accuracy: {acc:.4f}")
    return auc, acc

fit_and_score(X_leaky, y, "leaky")
fit_and_score(X_honest, y, "honest")

  leaky -> AUC: 0.9844  |  Accuracy: 0.9999
 honest -> AUC: 0.8950  |  Accuracy: 0.9999


(np.float64(0.8950245642170268), 0.9999222141444603)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation — substantial, non-random missingness on 3 of the 5 approved features.** Measured
directly in the availability query above: `ga4_total_engagement_sec` is NULL on 42.35% of
surviving rows, `word_count` on 33.35%, `backlinks` on 39.07% — while `gsc_clicks` and
`gsc_impressions` are essentially complete (0.00% NULL each). `dim_content` also carries
`provider_used`, `model_used`, `last_optimized_date`, and `optimization_eligible_date`,
pointing to a staged content-generation pipeline — the missingness likely tracks *pipeline stage*
(not yet GA4-instrumented, not yet AI-optimized, not yet backlink-crawled) rather than being
random. The feature frame built in Section 3 drops any row missing even one of these three
features, so it trains on a biased subset of pages, not the full population.

`dim_content` is confirmed (schema check below) to carry `content_updated_date` — pages do get
edited after publication — but it is joined at the content level only, one row per page, not one
row per `report_date`. So even where `word_count`/`backlinks` are populated, they reflect the
page's latest crawled state, not necessarily its true state on each historical date in March.

In [8]:
text_like_cols = [c for c in (fact_schema + dim_schema) if any(
    kw in c.lower() for kw in ["html", "body", "text", "content_text", "raw"]
)]
print("Columns matching a raw-text-like name:", text_like_cols if text_like_cols else "NONE FOUND")

dim_date_like_cols = [c for c in dim_schema if any(
    kw in c.lower() for kw in ["date", "updated", "snapshot", "version", "as_of", "_at"]
)]
print("dim_content columns:", dim_schema)
print("Date/versioning-like columns on dim_content:", dim_date_like_cols if dim_date_like_cols else "NONE FOUND")

Columns matching a raw-text-like name: NONE FOUND
dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
Date/versioning-like columns on dim_content: ['content_created_date', 'content_updated_date', 'keyword_created_date', 'last_optimized_date', 'optimization_eligible_date']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.